In [ ]:
#| hide
from nbdev_upc_aidl_iemocap_datasets.core import DatasetsFactory, AudioChunksDataset


# nbdev-upc-aidl-iemocap-datasets 🎧

> A convenience PyTorch Dataset factory for the [IEMOCAP emotional speech dataset](http://sail.usc.edu/iemocap/). This library provides seamless caching, on-the-fly log-Mel spectrogram feature extraction, SQLite-backed raw audio lookups, and notebook-friendly visualization tools.

[![Built with nbdev](https://img.shields.io/badge/Built%20with-nbdev-blue.svg)](https://nbdev.fast.ai/)
[![Package Manager: uv](https://img.shields.io/badge/package%20manager-uv-purple.svg)](https://github.com/astral-sh/uv)


> [!NOTE]
> **For contributors:** This file (`nbs/index.ipynb`) is the **source of truth** for the project's `README.md`.
> Do **not** edit `README.md` directly — it is auto-generated by running `nbdev_prepare` or `nbdev_export`.
> Any changes to the README content should be made here in `nbs/index.ipynb`.


## 📖 Overview

The **`nbdev-upc-aidl-iemocap-datasets`** package is a convenience module designed to simplify working with the **IEMOCAP (Interactive Emotional Dyadic Motion Capture)** speech dataset.

Rather than requiring manually downloaded audio archives and complex pre-processing scripts, this package provides a `DatasetsFactory` that downloads, caches, and indexes datasetchunks directly from a Cloudflare R2 bucket. It builds standard, high-performance PyTorch `Dataset` objects ready for Deep Learning (such as CNN/RNN classification models) with minimum configuration.

### ✨ Key Features

- **Automated R2 Remote Caching**: Seamlessly downloads and updates metadata, audio databases, and Parquet audio chunks from a remote **HTTPS** Cloudflare R2 bucket to a local cache directory (`~/.cache/upc-aidl-iemocap-datasets`). Downloads are handled by [`requests`](https://docs.python-requests.org/) for full SSL/TLS support.
- **SQLite Storage for Audio bytes**: Queries audio data on-demand from a single SQLite database, avoiding the overhead of managing thousands of individual audio files on your disk.
- **On-the-Fly Feature Extraction**: Dynamically reads, slices, zero-pads, and processes raw audio samples into log-scale Mel Spectrogram tensors using `librosa`.
- **Notebook-Friendly APIs**:
  - `get_dataset_audio_chunk_groups()` prints structured Pandas DataFrames.
  - `play_item_audio(idx)` lets you listen to sliced audio segments directly inside Jupyter/VS Code notebooks.
  - `plot_melspectrogram(idx)` renders a labelled log-Mel spectrogram plot chunk position, dominant emotion, and transcription) using `matplotlib`.
- **Built for Notebooks, Exported to Python**: Developed using `nbdev` and managed with `uv` for lightning-fast speeds and reliable developer experience.


## 🚀 Installation

Initially, this package is intended to be used directly via its public GitHub URL and is not published on PyPI.

You can install it into your environment using **`uv`**:

```sh
uv add "git+https://github.com/gofordiego/nbdev-upc-aidl-iemocap-datasets.git"
```

Or using standard `pip`:

```sh
pip install git+https://github.com/gofordiego/nbdev-upc-aidl-iemocap-datasets.git
```


### 📋 Key Runtime Dependencies

The package is installed with all dependencies automatically. The main ones are:

| Package | Purpose |
|---|---|
| `torch` / `torchaudio` | PyTorch `Dataset` & audio utilities |
| `librosa` | Log-Mel Spectrogram computation |
| `requests` | HTTPS downloads from Cloudflare R2 |
| `pandas` / `pyarrow` | Parquet audio chunk manifests |
| `soundfile` | Decoding in-memory WAV audio bytes |
| `tqdm` | Download progress bars |
| `matplotlib` | Spectrogram visualisation |
| `ipython` / `ipywidgets` | In-notebook audio playback widget |

## Quick-Start Usage

Point `url` to your Cloudflare R2 bucket (must be an **HTTPS** URL):

```python
from nbdev_upc_aidl_iemocap_datasets.core import DatasetsFactory

factory = DatasetsFactory(url="https://<your-r2-bucket>.r2.dev/")
factory.get_dataset_audio_chunk_groups()
```

### (Optional) Testing with local fixtures

`create_test_fixtures()` returns a `(tmpdir, tests_path)` pair populated with a stub SQLite DB, a stub Parquet audio chunk file, and a stub JSON manifest — all backed by a synthetic 440 Hz sine-wave audio clip.

The cells below use `create_test_fixtures()` so *this notebook* runs **without network access** and is safe to run in CI.

In [ ]:
from nbdev_upc_aidl_iemocap_datasets.test_utils import create_test_fixtures

tmpdir, tests_path = create_test_fixtures()
print('Fixtures ready at:', tests_path)

Fixtures ready at: /var/folders/bn/87jpkfs15bl3l7j40cpcr6140000gn/T/tmpk440mrpu


### Initialize the factory (no network)


In [ ]:
factory = DatasetsFactory(
    url="http://example.com/",
    # Use our tests fixtures path
    override_cache_dir=tests_path,
    # Prevent trying to refresh test cached files from example.com domain.
    refresh_json_file=False,
    refresh_sqlite_file=False,
)
factory.get_dataset_audio_chunk_groups()

,id,chunk_threshold_seconds,previous_overlap_seconds,sample_rate,last_export_filename,last_export_at,_parquet_path
0,1,4.8,0.2,16000,http://example.com/dataset_audio_chunks/group_...,2026-05-16 22:01:23.000000,/var/folders/bn/87jpkfs15bl3l7j40cpcr6140000gn...


### Build an `AudioChunksDataset`

Optional `partition_type`: "train", "validation", and "test".

In [ ]:
ds = factory.build_dataset(
    partition_type="test",  # Optional
    id=1,
    n_fft=1024,
    hop_length=160,
    win_length=400,
    n_mels=80,
    should_refresh_local_cache=False,
)
print(f'Dataset length: {len(ds)} chunks')


Dataset length: 3 chunks


### Inspect a single item

Each `ds[idx]` call returns a `(features, labels)` tuple:
- **`features`** — float32 tensor of shape `(n_mels, time_frames)` containing the log-Mel spectrogram.
- **`labels`** — float32 tensor of length 9 with per-emotion scores in the order `AudioChunksDataset.LABELS_EMOTIONS`.


In [ ]:
features, labels = ds[0]
print('features shape:', features.shape)   # (80, T)
print('labels shape: ', labels.shape)     # (9,)
print('emotions:', AudioChunksDataset.LABELS_EMOTIONS)
print('scores:  ', labels.tolist())


features shape: torch.Size([80, 481])
labels shape:  torch.Size([9])
emotions: ['frustrated', 'angry', 'sad', 'disgust', 'excited', 'fear', 'neutral', 'surprise', 'happy']
scores:   [0.6349999904632568, 0.3199999928474426, 0.006000000052154064, 0.006000000052154064, 0.006000000052154064, 0.006000000052154064, 0.006000000052154064, 0.006000000052154064, 0.006000000052154064]


### Notebook-Friendly APIs

`AudioChunksDataset` ships two interactive helpers designed for use inside Jupyter / VS Code notebooks.


#### `play_item_audio(idx)` — listen to a chunk in-notebook

Renders an `IPython.display.Audio` widget so you can play back the raw sliced (and optionally zero-padded) audio clip directly in the notebook without saving any files.


```python
# Plays the audio widget inline (works in Jupyter / VS Code notebooks)
ds.play_item_audio(0)
```


#### `plot_melspectrogram(idx)` — visualise a chunk in-notebook

Renders a labelled log-Mel spectrogram plot using `matplotlib`. Each plot shows thechunk position (e.g. * chunk 1 of 3*), the dominant emotion, and the utterance transcription as a super-title.


```python
# Renders a log-Mel spectrogram plot inline (works in Jupyter / VS Code notebooks)
ds.plot_melspectrogram(0)
```


## 🧠 Feature Engineering & PyTorch Dataset Details

### Feature Extraction
- **Features (`X`)**: Constructed as a Log-Mel Spectrogram PyTorch tensor.
  - Standard Sample Rate: `16,000 Hz`.
  - Max expected samples per slice is derived from the parameters: chunk_threshold_seconds` (e.g. `4.8s`) * `sample_rate` (e.g. `16,000`) = `76,800` samples.
  - If `should_add_padding` is `True`, the audio slice is 0-padded to the max sample limit before applying STFT and Mel scaling.
  - **Backend**: Log-Mel Spectrogram computation defaults to **`torchaudio`** ([`MelSpectrogram`](https://pytorch.org/audio/stable/generated/torchaudio.transforms.MelSpectrogram.html) + log scaling), which benchmarks ~**140% faster** than the equivalent `librosa` pipeline. `librosa` remains a fallback dependency for audio I/O.
- **Labels (`Y`)**: A sorted `FloatTensor` containing 9 emotion scores:
  1. frustrated
  2. angry
  3. sad
  4. disgust
  5. excited
  6. fear
  7. neutral
  8. surprise
  9. happy

### Caching Considerations
> [!TIP]
> **On-the-Fly Computation vs. In-Memory Caching:**
> Computing Log-Mel Spectrograms takes CPU power. While we do this on the fly inside `__getitem__` to avoid storing gigabytes of floating-point matrices in RAM, for fast training sessions you can enable an optional in-memory dictionary cache in `AudioChunksDataset` or store generated `.pt` tensors inside the cache directory.
> Prefetching features using `DataLoader(num_workers=4, pin_memory=True)` is also highly recommended.


## 🗄️ Database & File Schemas

### 1. SQLite Database Schema (`iemocap_dataset_table.db`)
This table holds the raw compressed audios and emotion ratings, indexed for microsecond-level retrieval.

```sql
CREATE TABLE iemocap_dataset (
        row_index INTEGER NOT NULL,
        compressed_audio_url VARCHAR,
        file VARCHAR,
        audio_bytes BLOB,
        frustrated FLOAT,
        angry FLOAT,
        sad FLOAT,
        disgust FLOAT,
        excited FLOAT,
        fear FLOAT,
        neutral FLOAT,
        surprise FLOAT,
        happy FLOAT,
        "EmoAct" FLOAT,
        "EmoVal" FLOAT,
        "EmoDom" FLOAT,
        gender VARCHAR,
        transcription VARCHAR,
        major_emotion VARCHAR,
        speaking_rate FLOAT,
        pitch_mean FLOAT,
        pitch_std FLOAT,
        rms FLOAT,
        relative_db FLOAT,
        speaker_id VARCHAR
        track_group_id VARCHAR
        track_group_speaker_turn INTEGER
        PRIMARY KEY (row_index)
);
CREATE UNIQUE INDEX ix_iemocap_dataset_file ON iemocap_dataset (file);
CREATE INDEX ix_iemocap_dataset_row_index ON iemocap_dataset (row_index);
```

### 2. Dataset Chunk Parquet Schema (`*.parquet`)
Eachchunk contains pre-segmentedchunks and alignment offsets.

```sql
id INTEGER NOT NULL,
dataset_audio_chunk_group_id VARCHAR,
source_dataset_table_name VARCHAR,
source_dataset_row_index INTEGER,
source_dataset_audio_filename VARCHAR,
chunk_compressed_audio_url VARCHAR,
source_duration_seconds FLOAT,
source_total_samples INTEGER,
total_chunk_parts INTEGER,
chunk_part_idx INTEGER,
start_sample_offset INTEGER,
end_sample_offset INTEGER,
sample_duration FLOAT,
should_add_padding BOOLEAN,
assigned_reviewer_id INTEGER,
last_reviewer_id INTEGER,
reviewed_at DATETIME,
should_exclude BOOLEAN,
reviewed_frustrated FLOAT,
reviewed_angry FLOAT,
reviewed_sad FLOAT,
reviewed_disgust FLOAT,
reviewed_excited FLOAT,
reviewed_fear FLOAT,
reviewed_neutral FLOAT,
reviewed_surprise FLOAT,
reviewed_happy FLOAT,
reviewed_major_emotion VARCHAR NOT NULL,
speaker_id VARCHAR,
track_group_id VARCHAR,
track_group_speaker_turn INTEGER
```


## 🛠️ Development Guidelines

This project uses modern tooling for high-speed local Python package management and notebook-integrated software development.

### 📦 1. Package Manager: `uv`
We use [**`uv`**](https://github.com/astral-sh/uv) to manage project dependencies and virtual environments. It is an extremely fast replacement for `pip`, `pip-tools`, and `poetry`.

- To install dependencies:
  ```bash
  uv sync
  ```
- To run commands inside the virtual environment:
  ```bash
  uv run <command>
  ```

### 📓 2. Development Flow: `nbdev`
This project is built using [**`nbdev`**](https://nbdev.fast.ai/), which allows us to write, test, document, and distribute Python packages directly inside Jupyter Notebooks.

> [!IMPORTANT]
> **Do not edit `.py` files inside `src/` directly!**
> All code logic must be written inside `.ipynb` notebooks located in the workspace. Running `nbdev` commands will export the notebook code into the Python modules automatically.

- **Export code from notebooks to library modules**:
  ```bash
  uv run nbdev-export
  ```
- **Run notebook-based tests**:
  ```bash
  uv run nbdev-test
  ```
- **Build the documentation website locally** (requires [Quarto](https://quarto.org/docs/get-started/)):
  ```bash
  uv run nbdev-preview
  ```
- **Prepare changes before committing** (runs export, clean, and tests all in one command):
  ```bash
  uv run nbdev-prepare
  ```
